In [ ]:

# If running in a fresh Colab runtime, uncomment the line below once:
# !pip -q install torch torchvision scikit-learn shap grad-cam opencv-python-headless pandas numpy matplotlib seaborn

import os, re, glob, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms, models

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, classification_report, cohen_kappa_score,
                              roc_auc_score, roc_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from PIL import Image
import cv2

import shap

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


In [ ]:

# ============================== CONFIG ==============================
# Root of the unzipped Kaggle X-ray archive. It must contain
# train/ val/ test/ auto_test/ each with subfolders 0/1/2/3/4.
ARCHIVE_DIR = "/content/archive(1)"          # <-- EDIT ME

# Path to the clinical CSV (the file you uploaded).
CLINICAL_CSV = "/content/clinical_info.csv"   # <-- EDIT ME

SPLITS = ["train", "val", "test", "auto_test"]
KL_GRADES = ["0", "1", "2", "3", "4"]

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 10          # bump this up once the pipeline is verified end-to-end
LR = 1e-4
# ======================================================================


In [ ]:

clinical_df = pd.read_csv(CLINICAL_CSV)
print("Clinical dataset shape:", clinical_df.shape)
clinical_df.head()


In [ ]:

print(clinical_df.dtypes)
print("\nMissing values per column:\n", clinical_df.isna().sum())
print("\nSIDE values:\n", clinical_df['SIDE'].value_counts())
print("\nSURGERY values:\n", clinical_df['SURGERY'].value_counts())
print("\nFREQUENT PAIN values:\n", clinical_df['FREQUENT PAIN'].value_counts())


In [ ]:

def scan_archive(archive_dir):
    rows = []
    for split in SPLITS:
        split_dir = os.path.join(archive_dir, split)
        if not os.path.isdir(split_dir):
            print(f"[WARN] split folder not found: {split_dir}")
            continue
        for grade in KL_GRADES:
            grade_dir = os.path.join(split_dir, grade)
            if not os.path.isdir(grade_dir):
                continue
            for fname in os.listdir(grade_dir):
                if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                    rows.append({
                        "SPLIT": split,
                        "KL_GRADE": int(grade),
                        "FILENAME": fname,
                        "IMAGE_PATH": os.path.join(grade_dir, fname)
                    })
    return pd.DataFrame(rows)

xray_df = scan_archive(ARCHIVE_DIR)
print("Total images found:", len(xray_df))
xray_df.groupby(["SPLIT", "KL_GRADE"]).size().unstack(fill_value=0)


In [ ]:

# Look at a handful of raw filenames per split -> this is what drives the regex below.
for split in SPLITS:
    sample = xray_df.loc[xray_df.SPLIT == split, "FILENAME"].head(5).tolist()
    print(split, "->", sample)


In [ ]:

ID_REGEX = re.compile(r"(\d+)")
LETTER_SIDE_REGEX = re.compile(r"\d+\D*([LlRr])")
SUFFIX_SIDE_REGEX = re.compile(r"_(\d)\.")

# EDIT THIS after inspecting your own filenames / screenshot if needed.
# Convention assumed here: suffix "_1" -> LEFT, "_2" -> RIGHT.
SUFFIX_SIDE_MAP = {"1": "LEFT", "2": "RIGHT"}

def extract_id(filename):
    m = ID_REGEX.search(filename)
    return int(m.group(1)) if m else None

def extract_side(filename):
    m = LETTER_SIDE_REGEX.search(filename)
    if m:
        return "LEFT" if m.group(1).upper() == "L" else "RIGHT"
    m = SUFFIX_SIDE_REGEX.search(filename)
    if m:
        return SUFFIX_SIDE_MAP.get(m.group(1), None)
    return None

xray_df["ID"] = xray_df["FILENAME"].apply(extract_id)
xray_df["SIDE"] = xray_df["FILENAME"].apply(extract_side)

print("Images with no parsable ID:", xray_df["ID"].isna().sum())
print("Images with no parsable SIDE:", xray_df["SIDE"].isna().sum())
xray_df.head()


In [ ]:

def build_master_dataset(xray_df, clinical_df):
    clin = clinical_df.copy()
    clin["ID"] = clin["ID"].astype(int)

    with_side = xray_df.dropna(subset=["SIDE"]).copy()
    merged_with_side = with_side.merge(clin, on=["ID", "SIDE"], how="inner", suffixes=("", "_clin"))

    # X-rays whose side we could not parse: fall back to ID-only match.
    no_side = xray_df[xray_df["SIDE"].isna()].copy()
    if len(no_side):
        # If a patient has exactly one clinical row for that ID, it's unambiguous.
        id_counts = clin["ID"].value_counts()
        unambiguous_ids = id_counts[id_counts == 1].index
        no_side_unambig = no_side[no_side["ID"].isin(unambiguous_ids)]
        merged_no_side = no_side_unambig.merge(clin, on="ID", how="inner", suffixes=("", "_clin"))
    else:
        merged_no_side = pd.DataFrame(columns=merged_with_side.columns)

    master = pd.concat([merged_with_side, merged_no_side], ignore_index=True, sort=False)
    return master, with_side, no_side

master_df, with_side_df, no_side_df = build_master_dataset(xray_df, clinical_df)
print("Matched X-rays (side-based):", len(with_side_df.merge(clinical_df, on=['ID','SIDE'])))
print("Total matched X-rays (incl. unambiguous ID-only fallback):", len(master_df))
print("Total X-rays found on disk:", len(xray_df))
print("Unmatched X-rays:", len(xray_df) - len(master_df))
master_df.head()


In [ ]:

matched_ids = set(zip(master_df["ID"], master_df.get("SIDE_clin", master_df["SIDE"])))
unmatched_clinical = clinical_df[~clinical_df.apply(lambda r: (r["ID"], r["SIDE"]) in matched_ids, axis=1)]

report = {
    "Total X-rays on disk": len(xray_df),
    "Matched X-rays": len(master_df),
    "Unmatched X-rays": len(xray_df) - len(master_df),
    "Unmatched clinical records": len(unmatched_clinical),
    "Duplicate clinical IDs (multi-row, i.e. both knees)": clinical_df["ID"].duplicated().sum(),
    "Duplicate ID+SIDE combos in clinical data": clinical_df.duplicated(subset=["ID", "SIDE"]).sum(),
    "Missing clinical values (any column)": int(clinical_df.isna().sum().sum()),
}
for k, v in report.items():
    print(f"{k}: {v}")

print("\nImages per KL grade (matched master dataset):")
print(master_df["KL_GRADE"].value_counts().sort_index())

print("\nSurgery cases in matched master dataset:")
print(master_df["SURGERY"].value_counts())

assert len(master_df) > 0, "STOP: mapping produced 0 matched rows - fix the regex/side logic above before continuing."


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.countplot(x="KL_GRADE", data=master_df, ax=axes[0], palette="viridis")
axes[0].set_title("KL Grade distribution (matched dataset)")

sns.histplot(clinical_df["AGE"], bins=20, kde=True, ax=axes[1])
axes[1].set_title("Age distribution")

sns.histplot(clinical_df["BMI"], bins=20, kde=True, ax=axes[2])
axes[2].set_title("BMI distribution")
plt.tight_layout(); plt.show()


In [ ]:

surgery_binary = master_df["SURGERY"].astype(str).str.startswith("1").astype(int)
plt.figure(figsize=(6, 4))
sns.countplot(x="KL_GRADE", hue=surgery_binary, data=master_df.assign(_surg=surgery_binary), palette="Set2")
plt.title("Surgery outcome vs. KL grade")
plt.legend(title="SURGERY", labels=["No", "Yes"])
plt.show()


In [ ]:

def check_patient_leakage(df):
    ids_by_split = df.groupby("SPLIT")["ID"].apply(set)
    leaked = False
    splits = list(ids_by_split.index)
    for i in range(len(splits)):
        for j in range(i + 1, len(splits)):
            overlap = ids_by_split[splits[i]] & ids_by_split[splits[j]]
            if overlap:
                leaked = True
                print(f"[WARN] {len(overlap)} patient IDs appear in both '{splits[i]}' and '{splits[j]}'")
    if not leaked:
        print("No patient-level leakage detected across the provided splits.")
    return leaked

leaked = check_patient_leakage(master_df)

if leaked:
    # Re-split at the patient level to guarantee no leakage: 70/15/15
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
    train_idx, temp_idx = next(gss1.split(master_df, groups=master_df["ID"]))
    temp_df = master_df.iloc[temp_idx]
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
    val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["ID"]))
    master_df.loc[master_df.index[train_idx], "SPLIT"] = "train"
    master_df.loc[temp_df.index[val_idx], "SPLIT"] = "val"
    master_df.loc[temp_df.index[test_idx], "SPLIT"] = "test"
    master_df = master_df[master_df["SPLIT"] != "auto_test"]  # drop stray auto_test after re-split
    print("Re-split at patient level. New split sizes:")

print(master_df["SPLIT"].value_counts())


In [ ]:

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(7),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class KneeXrayDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["IMAGE_PATH"]).convert("RGB")
        img = self.transform(img)
        label = int(row["KL_GRADE"])
        return img, label, idx

train_ds = KneeXrayDataset(master_df[master_df.SPLIT == "train"], train_transform)
val_ds   = KneeXrayDataset(master_df[master_df.SPLIT == "val"],   eval_transform)
test_ds  = KneeXrayDataset(master_df[master_df.SPLIT == "test"],  eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("train/val/test sizes:", len(train_ds), len(val_ds), len(test_ds))


In [ ]:

def build_resnet18(num_classes=5):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

resnet_model = build_resnet18().to(DEVICE)

# Class-weighted loss to handle KL-grade imbalance.
train_labels = master_df.loc[master_df.SPLIT == "train", "KL_GRADE"].values
class_weights = compute_class_weight(class_weight="balanced",
                                      classes=np.arange(5), y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print("Class weights:", class_weights)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(resnet_model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=2, factor=0.5)


In [ ]:

def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    torch.set_grad_enabled(is_train)
    for imgs, labels, _ in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        if is_train:
            optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        if is_train:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc

best_val_loss = float("inf")
history = []
for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(resnet_model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_epoch(resnet_model, val_loader, criterion, optimizer=None)
    scheduler.step(val_loss)
    history.append((epoch, tr_loss, tr_acc, val_loss, val_acc))
    print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} acc {tr_acc:.3f} "
          f"| val loss {val_loss:.4f} acc {val_acc:.3f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(resnet_model.state_dict(), "resnet18_kl_grade_best.pt")

resnet_model.load_state_dict(torch.load("resnet18_kl_grade_best.pt"))
print("Loaded best checkpoint (lowest val loss).")


In [ ]:

hist_df = pd.DataFrame(history, columns=["epoch", "train_loss", "train_acc", "val_loss", "val_acc"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_df.epoch, hist_df.train_loss, label="train"); axes[0].plot(hist_df.epoch, hist_df.val_loss, label="val")
axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(hist_df.epoch, hist_df.train_acc, label="train"); axes[1].plot(hist_df.epoch, hist_df.val_acc, label="val")
axes[1].set_title("Accuracy"); axes[1].legend()
plt.tight_layout(); plt.show()


In [ ]:

def get_predictions(model, loader):
    model.eval()
    preds, labels, indices = [], [], []
    with torch.no_grad():
        for imgs, lbls, idxs in loader:
            imgs = imgs.to(DEVICE)
            out = model(imgs)
            preds.extend(out.argmax(1).cpu().numpy())
            labels.extend(lbls.numpy())
            indices.extend(idxs.numpy())
    return np.array(preds), np.array(labels), np.array(indices)

kl_test_preds, kl_test_labels, kl_test_indices = get_predictions(resnet_model, test_loader)

kl_acc = accuracy_score(kl_test_labels, kl_test_preds)
kl_prec = precision_score(kl_test_labels, kl_test_preds, average="macro", zero_division=0)
kl_rec = recall_score(kl_test_labels, kl_test_preds, average="macro", zero_division=0)
kl_f1 = f1_score(kl_test_labels, kl_test_preds, average="macro", zero_division=0)
kl_qwk = cohen_kappa_score(kl_test_labels, kl_test_preds, weights="quadratic")

print(f"Accuracy:        {kl_acc:.3f}")
print(f"Macro Precision: {kl_prec:.3f}")
print(f"Macro Recall:    {kl_rec:.3f}")
print(f"Macro F1:        {kl_f1:.3f}")
print(f"Quadratic Weighted Kappa: {kl_qwk:.3f}")
print("\n", classification_report(kl_test_labels, kl_test_preds, digits=3))

cm = confusion_matrix(kl_test_labels, kl_test_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=KL_GRADES, yticklabels=KL_GRADES)
plt.xlabel("Predicted KL grade"); plt.ylabel("True KL grade"); plt.title("ResNet18 confusion matrix")
plt.show()


In [ ]:

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, input_tensor, class_idx=None):
        self.model.eval()
        output = self.model(input_tensor)
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()
        self.model.zero_grad()
        output[0, class_idx].backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam = cam.squeeze().cpu().numpy()
        cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx

gradcam = GradCAM(resnet_model, resnet_model.layer4[-1])

def show_gradcam(image_path, true_grade=None):
    img = Image.open(image_path).convert("RGB")
    input_tensor = eval_transform(img).unsqueeze(0).to(DEVICE)
    cam, pred_class = gradcam.generate(input_tensor)

    img_resized = np.array(img.resize((IMG_SIZE, IMG_SIZE))).astype(np.float32) / 255.0
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET) / 255.0
    heatmap = heatmap[..., ::-1]  # BGR -> RGB
    overlay = 0.55 * img_resized + 0.45 * heatmap
    overlay = np.clip(overlay, 0, 1)

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(img_resized); axes[0].set_title("Original X-ray"); axes[0].axis("off")
    axes[1].imshow(overlay); axes[1].set_title(f"Grad-CAM (pred KL={pred_class})"); axes[1].axis("off")
    if true_grade is not None:
        fig.suptitle(f"True KL grade: {true_grade}")
    plt.tight_layout(); plt.show()
    return pred_class

# Example: visualize a random test image
sample_row = master_df[master_df.SPLIT == "test"].sample(1, random_state=SEED).iloc[0]
show_gradcam(sample_row["IMAGE_PATH"], true_grade=sample_row["KL_GRADE"])


In [ ]:

def encode_clinical(df):
    df = df.copy()
    # SURGERY: "0: No" / "1: Yes" -> 0/1
    df["SURGERY_BIN"] = df["SURGERY"].astype(str).str.extract(r"^(\d)").astype(int)
    # FREQUENT PAIN: "0: No pain..." -> ordinal int 0-5
    df["FREQUENT_PAIN_ORD"] = df["FREQUENT PAIN"].astype(str).str.extract(r"^(\d)").astype(int)
    # Basic sanity clipping of obviously invalid numeric values
    for col in ["AGE", "HEIGHT", "WEIGHT", "MAX WEIGHT", "BMI"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=["AGE", "HEIGHT", "WEIGHT", "MAX WEIGHT", "BMI"])
    return df

master_df = encode_clinical(master_df)
master_df[["FREQUENT PAIN", "FREQUENT_PAIN_ORD", "SURGERY", "SURGERY_BIN"]].drop_duplicates().head(10)


In [ ]:

def predict_kl_for_all(model, df):
    ds = KneeXrayDataset(df, eval_transform)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    preds, _, idxs = get_predictions(model, loader)
    out = df.reset_index(drop=True).copy()
    out.loc[idxs, "PREDICTED_KL_GRADE"] = preds
    return out

master_df = predict_kl_for_all(resnet_model, master_df)
master_df["PREDICTED_KL_GRADE"] = master_df["PREDICTED_KL_GRADE"].astype(int)

FEATURE_COLS = ["PREDICTED_KL_GRADE", "AGE", "HEIGHT", "WEIGHT", "MAX WEIGHT", "BMI", "FREQUENT_PAIN_ORD"]
TARGET_COL = "SURGERY_BIN"

master_df[FEATURE_COLS + [TARGET_COL, "SPLIT"]].head()


In [ ]:

rf_train = master_df[master_df.SPLIT == "train"]
rf_val   = master_df[master_df.SPLIT == "val"]
rf_test  = master_df[master_df.SPLIT == "test"]

X_train, y_train = rf_train[FEATURE_COLS], rf_train[TARGET_COL]
X_val,   y_val   = rf_val[FEATURE_COLS],   rf_val[TARGET_COL]
X_test,  y_test  = rf_test[FEATURE_COLS],  rf_test[TARGET_COL]

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)

val_preds = rf_model.predict(X_val)
print("Validation accuracy:", accuracy_score(y_val, val_preds))


In [ ]:

rf_test_preds = rf_model.predict(X_test)
rf_test_probs = rf_model.predict_proba(X_test)[:, 1]

rf_acc = accuracy_score(y_test, rf_test_preds)
rf_prec = precision_score(y_test, rf_test_preds, zero_division=0)
rf_rec = recall_score(y_test, rf_test_preds, zero_division=0)   # sensitivity - important for a medical-risk model
rf_f1 = f1_score(y_test, rf_test_preds, zero_division=0)
rf_auc = roc_auc_score(y_test, rf_test_probs)

print(f"Accuracy:  {rf_acc:.3f}")
print(f"Precision: {rf_prec:.3f}")
print(f"Recall (sensitivity): {rf_rec:.3f}")
print(f"F1-score:  {rf_f1:.3f}")
print(f"ROC-AUC:   {rf_auc:.3f}")
print("\n", classification_report(y_test, rf_test_preds, digits=3))

cm = confusion_matrix(y_test, rf_test_preds)
plt.figure(figsize=(4, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges", xticklabels=["No surgery", "Surgery"],
            yticklabels=["No surgery", "Surgery"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Random Forest confusion matrix")
plt.show()

fpr, tpr, _ = roc_curve(y_test, rf_test_probs)
plt.figure(figsize=(4, 4))
plt.plot(fpr, tpr, label=f"AUC={rf_auc:.3f}")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.title("ROC curve"); plt.legend()
plt.show()


In [ ]:

explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test)

# For binary classification, shap_values is either a list [class0, class1] or a single 2D array
# depending on the SHAP version - handle both.
if isinstance(shap_values, list):
    sv_class1 = shap_values[1]
else:
    sv_class1 = shap_values

plt.figure()
shap.summary_plot(sv_class1, X_test, plot_type="bar", show=False)
plt.title("Global feature importance (mean |SHAP value|)")
plt.tight_layout(); plt.show()

plt.figure()
shap.summary_plot(sv_class1, X_test, show=False)
plt.tight_layout(); plt.show()


In [ ]:

# Local explanation for one individual patient in the test set
patient_idx = 0
patient_row = X_test.iloc[[patient_idx]]
patient_pred = rf_model.predict(patient_row)[0]
patient_prob = rf_model.predict_proba(patient_row)[0, 1]

print("Patient features:\n", patient_row)
print(f"\nPredicted SURGERY = {patient_pred} (probability = {patient_prob:.2%})")

shap.plots._waterfall.waterfall_legacy(
    explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) and len(np.atleast_1d(explainer.expected_value)) > 1 else explainer.expected_value,
    sv_class1[patient_idx],
    feature_names=X_test.columns.tolist(),
)


In [ ]:

def predict_patient(image_path, clinical_row, resnet_model, rf_model, explainer, feature_cols):
    # clinical_row: dict-like with AGE, HEIGHT, WEIGHT, MAX WEIGHT, BMI, FREQUENT_PAIN_ORD
    # --- Stage 1: KL grade from X-ray ---
    img = Image.open(image_path).convert("RGB")
    input_tensor = eval_transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = resnet_model(input_tensor)
        pred_kl = int(logits.argmax(1).item())

    # --- Late fusion feature vector ---
    feats = {"PREDICTED_KL_GRADE": pred_kl, **clinical_row}
    feat_df = pd.DataFrame([feats])[feature_cols]

    # --- Stage 3: Random Forest surgery prediction ---
    surgery_pred = int(rf_model.predict(feat_df)[0])
    surgery_prob = float(rf_model.predict_proba(feat_df)[0, 1])

    # --- Explanations ---
    pred_class = show_gradcam(image_path, true_grade=None)   # displays Grad-CAM plot
    shap_vals = explainer.shap_values(feat_df)
    sv = shap_vals[1][0] if isinstance(shap_vals, list) else shap_vals[0]
    contrib = sorted(zip(feature_cols, sv), key=lambda x: -abs(x[1]))

    print("=" * 40)
    print("KNEE OSTEOARTHRITIS ASSESSMENT")
    print("=" * 40)
    print(f"Predicted KL Grade: {pred_kl}")
    print(f"\nSurgery Risk/Outcome Prediction: {'YES' if surgery_pred == 1 else 'NO'}")
    print(f"Prediction Probability: {surgery_prob:.0%}")
    print("-" * 40)
    print("CLINICAL EXPLANATION (SHAP, ranked by |impact|)")
    print("-" * 40)
    for i, (feat, val) in enumerate(contrib, 1):
        direction = "increases" if val > 0 else "decreases"
        print(f"{i}. {feat}: {direction} surgery prediction (SHAP={val:+.3f})")
    print("=" * 40)
    return pred_kl, surgery_pred, surgery_prob, contrib

# Example usage on one real test-set patient:
example = rf_test.iloc[0]
clinical_input = {c: example[c] for c in ["AGE", "HEIGHT", "WEIGHT", "MAX WEIGHT", "BMI", "FREQUENT_PAIN_ORD"]}
predict_patient(example["IMAGE_PATH"], clinical_input, resnet_model, rf_model, explainer, FEATURE_COLS)


In [ ]:

summary_table = pd.DataFrame({
    "Task": ["KL Grade (ResNet18)"] * 5 + ["Surgery (Random Forest)"] * 5,
    "Metric": ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "QWK",
               "Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
    "Value": [kl_acc, kl_prec, kl_rec, kl_f1, kl_qwk, rf_acc, rf_prec, rf_rec, rf_f1, rf_auc],
})
summary_table["Value"] = summary_table["Value"].round(3)
summary_table
